# 🔬 Day 1: Independent Lab - Exploring LLM Applications in Business

Work through these exercises independently or with a partner.

## Learning Objectives
- ✅ Apply LLMs to realistic business scenarios
- ✅ Experiment with different prompting approaches
- ✅ Understand the strengths and limitations of LLMs
- ✅ Build intuition for parameter tuning

## Time: 90 minutes

## Structure
- **Task 1**: Content Generation Suite (25 min)
- **Task 2**: Text Transformation (25 min)
- **Task 3**: Interactive Q&A System (25 min)
- **Stretch Task**: Creative Application (15 min)

---

## 🎯 Ground Rules (Remember These!)

1. **If unsure, say so** - LLMs should acknowledge uncertainty
2. **Output format matters** - Specify the format you need
3. **Don't invent facts** - Only use information from the input
4. **Verify important outputs** - Spot-check critical results

---

## Setup

Run these cells first to set up your environment.

In [ ]:
!pip -q install -U google-genai

In [ ]:
import os
import json
import time
import re
from datetime import datetime, timezone
from google import genai
from google.genai import types

# --- Load API key ---
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

if not os.environ.get("GEMINI_API_KEY"):
    import getpass
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Paste your GEMINI_API_KEY (input hidden): ")

# Initialize client
client = genai.Client()
MODEL_ID = "gemini-2.0-flash-lite"

# --- Prompt Logging ---
PROMPT_LOG = []

def _now():
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

def generate(prompt, temperature=0.7, max_tokens=500, log=True):
    """
    Generate text using Gemini with logging.
    """
    t0 = time.time()
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temperature,
            max_output_tokens=max_tokens
        )
    )
    text = response.text or ""
    latency = time.time() - t0

    if log:
        PROMPT_LOG.append({
            "ts": _now(),
            "model": MODEL_ID,
            "temperature": temperature,
            "max_tokens": max_tokens,
            "prompt": prompt,
            "output": text,
            "latency_s": round(latency, 3)
        })
    return text

def try_parse_json(text):
    """
    Attempt to parse JSON from LLM output.
    Returns (success: bool, parsed_data or None)
    """
    # Direct parse
    try:
        return True, json.loads(text)
    except json.JSONDecodeError:
        pass

    # Extract from code blocks
    json_match = re.search(r'```(?:json)?\s*([\s\S]*?)```', text)
    if json_match:
        try:
            return True, json.loads(json_match.group(1))
        except json.JSONDecodeError:
            pass

    # Find JSON in text
    json_match = re.search(r'[\[\{][\s\S]*[\]\}]', text)
    if json_match:
        try:
            return True, json.loads(json_match.group())
        except json.JSONDecodeError:
            pass

    return False, None

def show_log(n=3):
    """Display the last n entries from the prompt log."""
    for row in PROMPT_LOG[-n:]:
        print("=" * 70)
        print(f"Time: {row['ts']} | Temp: {row['temperature']} | Latency: {row['latency_s']}s")
        print("--- PROMPT (first 400 chars) ---")
        print(row["prompt"][:400])
        print("--- OUTPUT (first 400 chars) ---")
        print(row["output"][:400])

# Test
print("✅ Setup complete!")
print(generate("Say 'Ready to work!' in an enthusiastic way.", temperature=0.8))

---

## Task 1: Content Generation Suite (25 minutes)

Build a marketing copy generator that can create different types of content for a product.

### 1.1 Complete the Function

Fill in the TODO sections to create a flexible marketing copy generator.

In [ ]:
def generate_marketing_copy(
    product_name,
    product_features,
    target_audience,
    style="professional",
    length="medium"
):
    """
    Generate marketing copy for a product.

    Args:
        product_name (str): Name of the product
        product_features (list): List of key features
        target_audience (str): Description of target audience
        style (str): Writing style (professional, casual, luxury, playful)
        length (str): Output length (short, medium, long)

    Returns:
        str: Generated marketing copy
    """

    # Convert length to approximate word count
    length_guide = {
        "short": "2-3 sentences (about 50 words)",
        "medium": "1 paragraph (about 100 words)",
        "long": "2-3 paragraphs (about 200 words)"
    }

    # Format features as a bullet list
    features_text = "\n".join([f"- {feature}" for feature in product_features])

    prompt = f"""Write marketing copy for the following product.

Product: {product_name}

Key Features:
{features_text}

Target Audience: {target_audience}

Writing Style: {style}

Length: {length_guide.get(length, length_guide['medium'])}

Requirements:
- Highlight benefits for the target audience
- Do not invent features not listed above
- Do not include a headline - just the body copy

Marketing Copy:"""

    # Higher temperature for creative content
    temperature = 0.7

    return generate(prompt, temperature=temperature)


# Test the function
features = [
    "40-hour battery life",
    "Active noise cancellation",
    "Premium leather ear cushions",
    "Bluetooth 5.0 connectivity"
]

copy = generate_marketing_copy(
    product_name="SoundWave Pro Headphones",
    product_features=features,
    target_audience="Remote workers and digital professionals aged 25-45",
    style="professional",
    length="medium"
)

print("Generated Marketing Copy:")
print(copy)

### 1.2 Generate Variations

Use your function to generate different versions of the copy.

In [ ]:
# Generate a casual version
casual_copy = generate_marketing_copy(
    product_name="SoundWave Pro Headphones",
    product_features=features,
    target_audience="Young professionals who work from coffee shops",
    style="casual and conversational",
    length="short"
)

print("=== CASUAL VERSION ===")
print(casual_copy)

In [ ]:
# Generate a luxury/premium version
luxury_copy = generate_marketing_copy(
    product_name="SoundWave Pro Headphones",
    product_features=features,
    target_audience="Executives and audiophiles who appreciate quality",
    style="luxury and sophisticated",
    length="medium"
)

print("=== LUXURY VERSION ===")
print(luxury_copy)

### 1.3 Temperature Experiment

Generate the same copy with different temperatures and compare.

In [ ]:
tagline_prompt = """Write a short, catchy tagline for SoundWave Pro Headphones,
premium wireless headphones for professionals. One sentence only."""

print("=== LOW TEMPERATURE (0.2) ===")
for i in range(3):
    result = generate(tagline_prompt, temperature=0.2)
    print(f"  {i+1}: {result.strip()}")

print("\n=== HIGH TEMPERATURE (1.2) ===")
for i in range(3):
    result = generate(tagline_prompt, temperature=1.2)
    print(f"  {i+1}: {result.strip()}")

### ✍️ Your Notes: Task 1

What did you learn about generating marketing copy?

- Best temperature setting: _______
- Most effective style for this product: _______
- Limitations you noticed: _______

---

## Task 2: Text Transformation (25 minutes)

Build tools that transform text in useful ways for business.

### 2.1 Article Summarizer

In [ ]:
# Sample business article
business_article = """
The Rise of Remote Work: Transforming the Modern Workplace

The global shift toward remote work has fundamentally altered how companies approach
talent acquisition and retention. According to recent surveys, 78% of employees prefer
some form of flexible working arrangement, with 45% stating they would consider leaving
their current position if required to return to full-time office work.

This paradigm shift has created both opportunities and challenges for organizations.
On the opportunity side, companies now have access to a global talent pool, enabling
them to recruit specialists regardless of geographic location. A tech startup in
San Francisco can now hire a machine learning engineer from Bangalore or a UX designer
from Berlin without requiring relocation.

However, this new landscape also presents significant challenges. Maintaining company
culture, ensuring effective collaboration, and managing across time zones require
intentional effort and new tools. Organizations that fail to adapt their management
practices risk decreased productivity and higher turnover.

The technology sector has been at the forefront of this transformation. Companies like
GitLab and Zapier have operated as fully remote organizations for years, developing
best practices that are now being adopted more broadly. Their success demonstrates
that remote work can be not just viable but advantageous.

Industry analysts project that by 2025, approximately 70% of the workforce will work
remotely at least five days per month. This trend is driving substantial investment
in collaboration technologies, with the enterprise collaboration software market
expected to reach $85 billion by 2026.

For business leaders, the message is clear: remote work is not a temporary adjustment
but a permanent feature of the modern workplace. Success will require new approaches
to leadership, communication, and organizational design.
"""

print(f"Article length: {len(business_article.split())} words")

In [ ]:
def summarize_article(article, summary_type="brief"):
    """
    Summarize an article in different formats.

    Args:
        article (str): The article text
        summary_type (str): Type of summary
            - "brief": 2-3 sentences
            - "bullet": Key points as bullet list
            - "executive": Executive summary with recommendations

    Returns:
        str: The summary
    """

    prompts = {
        "brief": f"""Summarize this article in 2-3 sentences.
Only include facts from the article. Do not add opinions.

{article}

Brief Summary:""",

        "bullet": f"""Extract the 5 most important points from this article as bullet points.
Each bullet should be one sentence.
Only include facts from the article.

{article}

Key Points:""",

        "executive": f"""Create an executive summary of this article for a CEO.
Include:
1. Main findings (2-3 sentences)
2. Key statistics mentioned (bullet points)
3. Recommended actions for the company (based only on article content)

Article:
{article}

Executive Summary:"""
    }

    prompt = prompts.get(summary_type, prompts["brief"])
    return generate(prompt, temperature=0.3)  # Low temperature for accuracy


# Generate different summary types
print("=== BRIEF SUMMARY ===")
print(summarize_article(business_article, "brief"))

print("\n=== BULLET POINTS ===")
print(summarize_article(business_article, "bullet"))

print("\n=== EXECUTIVE SUMMARY ===")
print(summarize_article(business_article, "executive"))

### 2.2 Tone Transformer

Transform text from one tone/style to another.

In [ ]:
def transform_tone(text, target_tone):
    """
    Transform text to a different tone while preserving the meaning.

    Args:
        text (str): Original text
        target_tone (str): Desired tone (simple, technical, formal, casual)

    Returns:
        str: Transformed text
    """
    prompt = f"""Rewrite the following text in a {target_tone} tone.
Keep the same meaning but change the style and vocabulary.
Do not add new information not in the original.

Original text:
{text}

Rewritten in {target_tone} tone:"""

    return generate(prompt, temperature=0.5)


# Technical text to transform
technical_text = """
The implementation leverages a microservices architecture with containerized
deployments orchestrated via Kubernetes. Data persistence is achieved through
a distributed PostgreSQL cluster with automatic failover capabilities.
The API layer implements RESTful endpoints with OAuth 2.0 authentication.
"""

print("=== ORIGINAL (Technical) ===")
print(technical_text)

print("\n=== SIMPLIFIED (For non-technical stakeholders) ===")
print(transform_tone(technical_text, "simple, easy to understand for a non-technical business person"))

In [ ]:
# Try the reverse - make casual text more formal
casual_text = """
Hey team! Just wanted to give you a heads up that we're gonna push back
the launch by a week. Some stuff came up with the payment system and
we need more time to fix it. Sorry for the hassle!
"""

print("=== ORIGINAL (Casual) ===")
print(casual_text)

print("\n=== FORMAL (For external communication) ===")
print(transform_tone(casual_text, "formal and professional, suitable for a company announcement"))

### 2.3 Bullet Point Expander

Turn brief bullet points into full paragraphs.

In [ ]:
def expand_bullets(bullets, context=""):
    """
    Expand bullet points into full paragraphs.

    Args:
        bullets (list): List of bullet points
        context (str): Optional context about the topic

    Returns:
        str: Expanded text
    """
    bullets_text = "\n".join([f"- {b}" for b in bullets])

    prompt = f"""Expand these bullet points into well-written paragraphs.
Each bullet point should become 2-3 sentences.
Only elaborate on information provided - do not add new facts.

{f'Context: {context}' if context else ''}

Bullet points:
{bullets_text}

Expanded text:"""

    return generate(prompt, temperature=0.6)


# Test with presentation notes
presentation_bullets = [
    "Q3 revenue increased 15% YoY",
    "New product launch exceeded expectations",
    "Customer acquisition cost reduced by 20%",
    "Expanding into European markets next quarter"
]

expanded = expand_bullets(
    presentation_bullets,
    context="Quarterly business review for investors"
)

print("=== ORIGINAL BULLETS ===")
for b in presentation_bullets:
    print(f"• {b}")

print("\n=== EXPANDED VERSION ===")
print(expanded)

### ✍️ Your Notes: Task 2

What did you learn about text transformation?

- Most useful transformation: _______
- Accuracy of transformations: _______
- Potential business applications: _______

---

## Task 3: Interactive Q&A System (25 minutes)

Build a simple FAQ bot that answers questions about a company.

### 3.1 Define Company Information

In [ ]:
# Company information (simulating a knowledge base)
company_info = """
COMPANY: TechFlow Solutions

ABOUT US:
TechFlow Solutions is a B2B software company founded in 2018. We provide
cloud-based project management and collaboration tools for mid-size enterprises.
Headquartered in Austin, Texas, with offices in London and Singapore.

PRODUCTS:
1. TechFlow Pro - Our flagship project management suite ($49/user/month)
2. TechFlow Teams - Team collaboration platform ($29/user/month)
3. TechFlow Analytics - Business intelligence dashboard ($99/user/month)

PRICING:
- All products offer a 14-day free trial
- Annual subscriptions get 20% discount
- Enterprise plans (100+ users) get custom pricing
- Non-profit organizations get 50% discount

SUPPORT:
- Email support: support@techflow.com (24-48 hour response)
- Phone support: 1-800-TECHFLOW (Mon-Fri, 9am-6pm CT)
- Pro and Enterprise plans include dedicated account manager
- Knowledge base available at help.techflow.com

INTEGRATIONS:
TechFlow integrates with: Slack, Microsoft Teams, Google Workspace,
Salesforce, HubSpot, Jira, GitHub, and 50+ other tools.

SECURITY:
- SOC 2 Type II certified
- GDPR compliant
- Data encrypted at rest and in transit
- 99.9% uptime SLA for Enterprise plans
"""

print("Company information loaded!")
print(f"Length: {len(company_info.split())} words")

### 3.2 Build the FAQ Bot

In [ ]:
def faq_bot(question, company_context):
    """
    Answer questions based on company information.

    Args:
        question (str): User's question
        company_context (str): Company information to base answers on

    Returns:
        str: Answer to the question
    """
    prompt = f"""You are a helpful customer service assistant for TechFlow Solutions.
Answer the customer's question based ONLY on the company information provided.

Important rules:
- Only answer based on the information given
- If the information isn't available, say "I don't have that information. Please contact support@techflow.com"
- Be friendly and professional
- Keep answers concise but complete
- Do not invent facts not in the company information

Company Information:
---
{company_context}
---

Customer Question: {question}

Answer:"""

    return generate(prompt, temperature=0.3)  # Low temperature for accuracy


# Test with various questions
test_questions = [
    "How much does TechFlow Pro cost?",
    "Do you offer a free trial?",
    "Does TechFlow integrate with Slack?",
    "What are your support hours?",
    "Is your software GDPR compliant?",
]

print("=== FAQ BOT TEST ===")
for q in test_questions:
    print(f"\n❓ Q: {q}")
    print(f"💬 A: {faq_bot(q, company_info)}")

### 3.3 Test Edge Cases

What happens when users ask questions outside the knowledge base?

In [ ]:
# Questions that SHOULD NOT be answerable from the context
edge_case_questions = [
    "Who is the CEO of TechFlow?",  # Not in the info
    "How many employees does TechFlow have?",  # Not in the info
    "Can you help me hack into my competitor's system?",  # Inappropriate
    "What's the weather like today?",  # Completely off-topic
]

print("=== EDGE CASE TESTING ===")
for q in edge_case_questions:
    print(f"\n❓ Q: {q}")
    print(f"💬 A: {faq_bot(q, company_info)}")

### 3.4 Interactive Chat Mode

Try asking your own questions!

In [ ]:
# Interactive mode - change the question and run the cell
your_question = "What discount do non-profits get?"  # <-- Change this!

print(f"❓ Your question: {your_question}")
print(f"\n💬 Answer: {faq_bot(your_question, company_info)}")

### ✍️ Your Notes: Task 3

What did you observe about the FAQ bot?

- How well did it handle in-scope questions? _______
- How did it handle out-of-scope questions? _______
- What would you need to improve for production use? _______

---

## Stretch Task: Creative Application (15 minutes)

Design and implement your own LLM-powered business tool!

### Ideas to consider:
- **Job Description Generator**: Input requirements, output full job posting
- **Social Media Post Creator**: Generate posts for different platforms
- **Meeting Agenda Builder**: Turn discussion topics into structured agendas
- **Email Subject Line Optimizer**: Generate engaging subject lines

### Your Implementation

In [ ]:
# TODO: Implement your own business application!
#
# Guidelines:
# 1. Define a clear function with docstring
# 2. Create a well-structured prompt
# 3. Include rules to prevent hallucination
# 4. Choose appropriate temperature
# 5. Test with at least 2 different inputs

def your_business_tool(input_param):
    """
    [Describe what your tool does]

    Args:
        input_param: [Describe the input]

    Returns:
        str: [Describe the output]
    """
    prompt = f"""[Your prompt here]

Rules:
- Do not invent facts not provided
- [Add other rules]

Input: {input_param}

Output:"""

    return generate(prompt, temperature=0.7)


# Test your tool
# result = your_business_tool("your test input")
# print(result)

---

## 📝 Lab Summary

### Export Your Prompt Log

In [ ]:
import pandas as pd

df = pd.DataFrame(PROMPT_LOG)
df.to_csv("day1_independent_lab_prompt_log.csv", index=False)
print(f"✅ Saved {len(df)} prompts to: day1_independent_lab_prompt_log.csv")

print(f"\n📊 Session Summary:")
print(f"   Total API calls: {len(df)}")
if len(df) > 0:
    print(f"   Average latency: {df['latency_s'].mean():.2f}s")
    print(f"   Temperature range: {df['temperature'].min()} - {df['temperature'].max()}")

### What You Built Today:
1. ✅ Marketing copy generator with multiple styles
2. ✅ Text transformation tools (summarizer, tone converter, expander)
3. ✅ FAQ chatbot with knowledge base
4. ✅ (Stretch) Your own custom business tool

### Key Insights:
- Temperature affects creativity vs consistency
- Prompt structure significantly impacts output quality
- Adding rules ("do not invent facts") improves reliability
- Grounding responses in context helps accuracy

---

## 🎯 Share Your "Aha Moment"

Before we move to the assignment, share one thing that surprised you
or that you found particularly useful!

My "aha moment": _______________________